# BirdCLEF+ 2025 - Pretrained models, A2: Perch 2.0 + head, inference (submission notebook)

**CPU only, internet off.** This notebook uses the same protocol as `infer-v2`: twelve 5 s windows per 1-minute soundscape, with the same `row_id` and columns. Four worker processes each handle whole files, with one ONNX Runtime thread each.

It needs this competition's data, `perch_v2.onnx` from the private dataset `strippedbanana/sc4000-birdclef-m3-perch-onnx`, and the head from the A2 embed notebook. When `test_soundscapes/` is empty, it does a dry run on training soundscapes and projects 700 files against the 75 minute budget.

In [ ]:
import os
for v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[v] = "1"                # set before numpy loads; each worker uses one core

import glob, hashlib, importlib.metadata, json, subprocess, sys, time
from multiprocessing import Pool
from pathlib import Path

import numpy as np
import pandas as pd

T_NOTEBOOK = time.time()
SMOKE_TEST = os.environ.get("SMOKE_TEST") == "1"     # local test only, never set on Kaggle
ORT_VERSION = "1.31.1"
NUM_WORKERS = int(os.environ.get("M3_INFER_WORKERS", "4"))      # 0 = no Pool (Windows tests)
DRY_FILES = int(os.environ.get("M3_DRY_FILES", "20"))
BUDGET_MIN = 75

def find_parent(marker, env, fallback):
    if os.environ.get(env):
        return Path(os.environ[env])
    hits = glob.glob(f"/kaggle/input/**/{marker}", recursive=True)
    return Path(hits[0]).parent if hits else Path(fallback)

DATA = find_parent("birdclef-2025/sample_submission.csv", "M3_DATA_DIR", "../../data")
HEAD_NAME = "perch_head_coral_std.npz"          # exact file from coral-heads, never the embed kernel's perch_head.npz
HEAD_DIR = find_parent(HEAD_NAME, "M3_MODEL_DIR", "../coral_heads")
if not (HEAD_DIR / HEAD_NAME).is_file():
    raise FileNotFoundError(f"{HEAD_NAME} not found in {HEAD_DIR}: attach strippedbanana/sc4000-birdclef-m3-coral-heads")
ONNX_PATH = Path(os.environ["M3_PERCH_ONNX"]) if os.environ.get("M3_PERCH_ONNX") \
    else find_parent("perch_v2.onnx", "M3_ONNX_DIR", ".") / "perch_v2.onnx"
assert ONNX_PATH.exists(), f"perch_v2.onnx not found: attach strippedbanana/sc4000-birdclef-m3-perch-onnx ({ONNX_PATH})"

# Only install from the attached wheels if the image lacks this exact version. Nothing imports
# onnxruntime in this process, so the workers are the first to load it.
try:
    have = importlib.metadata.version("onnxruntime")
except importlib.metadata.PackageNotFoundError:
    have = None
wheels = find_parent("ort_cpu_wheels", "M3_WHEELS_DIR", "../a2_perch_embed") / "ort_cpu_wheels"   # the embed kernel, not the heads kernel
if have != ORT_VERSION and not SMOKE_TEST:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-index", "--no-deps", "--force-reinstall",
                        "--find-links", str(wheels), f"onnxruntime=={ORT_VERSION}"], capture_output=True, text=True)
    if r.returncode != 0:
        if have is None:
            raise RuntimeError(f"onnxruntime is missing and no wheel fits this Python: {r.stderr[-400:]}")
        print(f"Keeping image onnxruntime {have}; wheel install failed: {r.stderr[-200:]}")
print("onnxruntime image version:", have, "| wanted:", ORT_VERSION)

species = json.load(open(HEAD_DIR / "species.json"))
sample_sub = pd.read_csv(DATA / "sample_submission.csv")
assert list(sample_sub.columns[1:]) == species, "species order differs from sample_submission"
print("Data:", DATA, "| head:", HEAD_DIR, "| onnx:", ONNX_PATH, "| CPUs:", os.cpu_count(), "| workers:", NUM_WORKERS)
print("Head:", json.load(open(HEAD_DIR / "head_cfg.json"))["best_head"])

In [ ]:
if not SMOKE_TEST:
    h = hashlib.sha256()
    with open(ONNX_PATH, "rb") as f:
        for block in iter(lambda: f.read(1 << 24), b""):
            h.update(block)
    if h.hexdigest() != "bf0c8467a924cb074663970ca4a0ab1e143602121930209657d0dff5d5cefa1f":
        raise RuntimeError(f"perch_v2.onnx checksum mismatch: {h.hexdigest()}")

test_files = sorted((DATA / "test_soundscapes").glob("*.ogg"))
DRY_RUN = len(test_files) == 0
if DRY_RUN:
    test_files = sorted((DATA / "train_soundscapes").glob("*.ogg"))[:DRY_FILES]
print(f"{'DRY RUN on' if DRY_RUN else 'Predicting'} {len(test_files)} soundscapes")

## Worker code (one ONNX session per process, no Perch logits used)

In [ ]:
import os, time
import numpy as np
import soundfile as sf
from scipy.signal import resample_poly

SR, WIN_SEC = 32_000, 5
WIN = SR * WIN_SEC          # 160000 samples, the fixed Perch 2.0 input
PERCH_SHA256 = "bf0c8467a924cb074663970ca4a0ab1e143602121930209657d0dff5d5cefa1f"

def to_32k_mono(y, sr):
    """Shared with the other A2 notebook, keep both copies identical."""
    if y.ndim > 1:
        y = y.mean(axis=1)
    if sr != SR:
        y = resample_poly(y, SR, sr)     # polyphase, as perch-hoplite does
    return y.astype(np.float32)

def preprocess_windows(x):
    """Perch preprocessing, per window: remove the mean, scale the peak to 0.25.
    Shared with the other A2 notebook, keep both copies identical."""
    x = x - x.mean(axis=1, keepdims=True)
    peak = np.abs(x).max(axis=1, keepdims=True)
    return (x * (0.25 / np.maximum(peak, 1e-8))).astype(np.float32)

def head_predict(head, emb):
    """Sigmoid probabilities from saved head arrays. Shared with the other A2 notebook."""
    x = (emb - head["mean"]) / head["std"]
    n_layers = int(head["n_layers"])
    for i in range(n_layers):
        x = x @ head[f"W{i}"].T + head[f"b{i}"]
        if i < n_layers - 1:
            x = np.maximum(x, 0.0)       # ReLU
    return 1.0 / (1.0 + np.exp(-x))

_sess, _head, _init_s = None, None, 0.0

def init_worker(onnx_path, head_path):
    global _sess, _head, _init_s
    t0 = time.perf_counter()
    import onnxruntime as ort          # first load happens inside the worker
    so = ort.SessionOptions()
    so.intra_op_num_threads = 1
    so.inter_op_num_threads = 1
    so.execution_mode = ort.ExecutionMode.ORT_SEQUENTIAL
    _sess = ort.InferenceSession(str(onnx_path), so, providers=["CPUExecutionProvider"])
    assert _sess.get_inputs()[0].name == "inputs"
    _head = dict(np.load(head_path))
    _init_s = time.perf_counter() - t0

def predict_file(path):
    t0 = time.perf_counter()
    y, sr = sf.read(str(path), dtype="float32")
    y = to_32k_mono(y, sr)
    t1 = time.perf_counter()
    n_win = max(1, int(np.ceil(len(y) / WIN)))
    y = np.pad(y, (0, n_win * WIN - len(y)))
    x = preprocess_windows(np.stack([y[i * WIN:(i + 1) * WIN] for i in range(n_win)]))
    t2 = time.perf_counter()
    emb = _sess.run(["embedding"], {"inputs": x})[0]       # the 12 windows of a file are one batch
    t3 = time.perf_counter()
    p = head_predict(_head, emb).astype(np.float32)
    t4 = time.perf_counter()
    # row_id = <file stem>_<end time in seconds>, e.g. soundscape_8358733_5
    row_ids = [f"{path.stem}_{(i + 1) * WIN_SEC}" for i in range(n_win)]
    return row_ids, p, dict(decode=t1 - t0, preprocess=t2 - t1, perch=t3 - t2, head=t4 - t3,
                            init=_init_s, pid=os.getpid())

## Predict, write `submission.csv`, report timing

In [ ]:
t0 = time.time()
initargs = (ONNX_PATH, HEAD_DIR / HEAD_NAME)
if NUM_WORKERS > 0:
    with Pool(NUM_WORKERS, initializer=init_worker, initargs=initargs) as pool:
        results = list(pool.imap(predict_file, test_files, chunksize=1))
else:
    init_worker(*initargs)
    results = [predict_file(p) for p in test_files]
wall = time.time() - t0

row_ids = [r for ids, _, _ in results for r in ids]
preds = np.concatenate([p for _, p, _ in results])
sub = pd.DataFrame(preds, columns=species)
sub.insert(0, "row_id", row_ids)
sub.to_csv("submission.csv", index=False)
print(sub.shape); print(sub.iloc[:3, :4])

stage = lambda k: sum(r[2][k] for r in results)
inits = {r[2]["pid"]: r[2]["init"] for r in results}
init_wall = max(inits.values())
per_file = max(wall - init_wall, 0.0) / len(test_files)
projected = (init_wall + per_file * 700) / 60
total = time.time() - T_NOTEBOOK
print(f"TIMING | decode {stage('decode'):.1f}s | preprocess {stage('preprocess'):.1f}s | "
      f"perch {stage('perch'):.1f}s | head {stage('head'):.2f}s (summed over {len(inits)} workers)")
print(f"TIMING | model load per worker {init_wall:.1f}s | predict wall {wall:.1f}s | {len(test_files)} files | "
      f"{per_file:.2f}s per file")
print(f"TIMING | projected 700 files: {projected:.1f} min (budget {BUDGET_MIN}) -> "
      f"{'OK' if projected <= BUDGET_MIN else 'OVER BUDGET'}")
print(f"TOTAL notebook time: {total / 60:.2f} min")